# Storage Layout and Logging

## What you'll learn

- The three-path model: delta, staging, and working directories
- What's inside the runs directory after a pipeline completes
- Delta Lake table structure and what each table contains
- Working directory defaults and when to preserve them
- Configuring logging: level, file handler, noise suppression

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb)  
**Estimated time:** 15 minutes  
**GPU required:** No.

---

Every pipeline run produces files in three directory trees. Understanding
this layout helps you debug failures, manage disk space, and configure
paths for your environment.

In [ ]:
from __future__ import annotations

import os
import tempfile

import polars as pl

from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.schemas import TablePath
from artisan.utils import configure_logging, tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

## The three-path model

Every pipeline is configured with three root directories:

| Path | Purpose | Lifetime |
|------|---------|----------|
| `delta_root` | Delta Lake tables — artifacts, metadata, provenance | Permanent |
| `staging_root` | Worker output staging area before commit | Temporary (cleared after commit) |
| `working_root` | Sandbox directories where operations execute | Temporary (cleaned after each execution) |

The **delta root** is the pipeline's durable store. All artifacts,
execution records, and provenance edges live here as Delta Lake tables.

The **staging root** is scratch space where workers write outputs
before they're committed to the delta store. After a successful
commit, staging files are removed.

The **working root** is where each operation's sandbox directory is
created. Operations write intermediate files here during execution.
By default, sandboxes are cleaned up after each execution completes.

In [ ]:
env = tutorial_setup("storage_layout")

print(f"delta_root:   {env.delta_root}")
print(f"staging_root: {env.staging_root}")
print(f"working_root: {env.working_root}")
print(f"\nAll under: {env.runs_dir}")

## Running a pipeline and exploring the output

Let's run a 3-step pipeline and inspect the directory tree afterward.

In [ ]:
pipeline = PipelineManager.create(
    name="storage_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

result = pipeline.finalize()
print(f"Pipeline complete: {result['total_steps']} steps")

In [ ]:
def show_tree(path: str, prefix: str = "", max_depth: int = 2, depth: int = 0) -> None:
    """Display a directory tree up to max_depth."""
    if depth >= max_depth or not os.path.isdir(path):
        return
    entries = sorted(os.path.join(path, name) for name in os.listdir(path))
    for i, entry in enumerate(entries):
        connector = (
            "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        )
        suffix = "/" if os.path.isdir(entry) else ""
        print(f"{prefix}{connector}{os.path.basename(entry)}{suffix}")
        if os.path.isdir(entry):
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            show_tree(entry, prefix + extension, max_depth, depth + 1)


print(f"{os.path.basename(env.runs_dir)}/")
show_tree(env.runs_dir, max_depth=3)

Key observations:

- **`delta/`** contains the Delta Lake tables — this is the permanent record
- **`staging/`** has no remaining payloads for these successful steps — their files were cleaned up after verified commitment
- **`working/`** is empty — sandbox directories were cleaned up after execution
- **`logs/runs/`** contains a separate directory and `pipeline.log` for each manager session

## Inspect the stored tables

The artifact index records identities and types. Content tables hold each
artifact type’s data, provenance tables hold relationships, and orchestration
tables record steps and executions.

Print the framework table paths and see which were created by this run:

In [ ]:
for tp in TablePath:
    table_dir = os.path.join(env.delta_root, tp)
    exists = "\u2713" if os.path.exists(table_dir) else "\u2717"
    print(f"  {exists} {tp.value}")

`inspect_pipeline` and `inspect_step` read committed results. They apply
Artisan’s visibility rules so an incomplete persistence attempt does not appear
as accepted output:

In [ ]:
inspect_pipeline(env.delta_root)

In [ ]:
inspect_step(env.delta_root, step_number=0)

For **physical storage diagnostics**, you can read a Delta table directly with
Polars. This completed tutorial run is safe to inspect, but `pl.read_delta` does
not apply Artisan’s logical commit checks. After a crash it can include rows
from unfinished writes. Use `inspect_step`, `ArtifactStore`, and the other
[committed readers](../../how-to-guides/inspecting-provenance.md) for application
results.

In [ ]:
artifact_index = pl.read_delta(os.path.join(env.delta_root, TablePath.ARTIFACT_INDEX))
print(artifact_index.head())

## Working directory defaults

When you create a pipeline without specifying `working_root`, it
defaults to the system temp directory. This is intentional — sandbox
directories are ephemeral scratch space.

In [ ]:
print(f"Default working_root: {tempfile.gettempdir()}")
print(f"Tutorial working_root: {env.working_root}")

For debugging, two flags control cleanup:

- `preserve_working=True` — keep sandbox directories after execution
  so you can inspect intermediate files
- `preserve_staging=True` — keep staged Parquet files after commitment so you can
  inspect the worker's original payloads

Pass these to `PipelineManager.create()` when debugging.

Uncommitted staging is retained after interruption or cancellation regardless of
this flag. Startup recovery can commit finished work before cache lookup; see
[Staging preservation](../../concepts/storage-and-delta-lake.md#staging-preservation).

## Configuring logging

A local pipeline manager owns a session log from creation through `finalize()`.
Read `pipeline.log_path` to find it, including after the run finishes. Resuming a
run creates a new session log.

`configure_logging` sets the process-wide severity and console behavior. Use
`level="DEBUG"` to include more detail; `level="INFO"` is enough for this example.

In [ ]:
# configure_logging is idempotent — safe to call multiple times
configure_logging(level="INFO", suppress_noise=True)

The `suppress_noise=True` default sets noisy third-party loggers
(httpx, httpcore, asyncio) to `CRITICAL` level. Without
this, debug output is dominated by HTTP client chatter.

In [ ]:
# The owned path stays available after pipeline.finalize().
log_path = pipeline.log_path
print(f"Session log: {log_path}")
if log_path is not None:
    with open(log_path) as f:
        lines = f.read().splitlines()
    print(f"Pipeline log: {len(lines)} lines")
    print("\n".join(lines[-5:]))

The session log contains this manager’s Artisan orchestration messages. Worker
stdout and unrelated application logs have separate destinations. Finalize the
manager to flush and close its log. Cloud storage does not create this local
file, so `pipeline.log_path` is `None` there.

For failed executions, use `inspect_failures` to find their failure reports.
[Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb)
shows how to read those reports.

## Crash recovery

After interruption, startup retries eligible recorded commits and recovers
validated finished execution units in batches by source step before cache lookup. First confirm that the
old orchestrator has stopped; only one driver or repair process may write a
store. To inspect retained evidence separately, request a repair report:

In [ ]:
print(
    "artisan store repair --delta-root runs/delta --staging-root runs/staging "
    "--recover-staging"
)

Report mode audits historical commit effects without changing the store. Add
`--apply` to recover validated work, and `--preserve-staging` to keep its original
payloads after commitment. Applied recovery checks its affected batches; it does
not repeat the historical audit. Rows
remain invisible **to Artisan’s committed readers** until their logical commit
is complete. Direct Delta reads may already show unfinished physical rows. See
[Repair a Store](../../how-to-guides/configuring-execution.md#recovering-from-crashes)
for eligibility, unresolved evidence, and explicit abandonment.

## Summary

You inspected the persistent store, temporary staging and working directories,
and the manager’s session log. Use committed readers for results and raw Delta
reads when diagnosing physical storage. Preserve working or staging files when
you need to investigate a failed execution.

## Next steps

- [Storage and Delta Lake](../../concepts/storage-and-delta-lake.md) — Storage and visibility
- [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md) — Store results remotely
- [Error Handling in Practice](../05-errors-and-control/02-error-visibility.ipynb) — Read failure reports
- [Resume and Caching](../03-caching/01-resume-and-caching.ipynb) — Reuse persisted work